# SVR (Support Vector Regression)

보통 회귀 모델은 모든 오차를 줄이려고 한다. 하지만 SVR은 조금 다르게 생각한다.

**이 정도 오차는 괜찮다고 보고, 일정 범위 안의 오차는 크게 문제 삼지 말자.**

이 허용 범위가 바로 `epsilon(ε)` 이다.

즉, SVR은
- 작은 오차는 어느 정도 허용하고
- 그 범위를 벗어난 오차에 더 집중하며
- 필요하면 커널을 사용해 비선형 관계도 학습하는 회귀 모델이다.

## 핵심 용어
- epsilon(ε): 이 정도 오차는 괜찮다고 보는 허용 범위
- C: 허용 범위를 벗어난 오차를 얼마나 엄격하게 벌줄지 정하는 값
- kernel: 직선으로 표현할지, 곡선까지 허용할지 정하는 방식

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 01. SVR 감 잡기

먼저 곡선 형태의 데이터를 만들어서
- 선형 커널
- 다항 커널
- RBF 커널

이 세 가지가 어떻게 다른 모양으로 예측하는지 확인한다.

## 02. 보스턴 집 값 회귀 문제

실제 회귀 데이터셋에 SVR을 적용해 본다.

1. SVR은 거리 개념을 사용하므로 스케일링이 매우 중요하다.
2. 커널에 따라 성능이 달라질 수 있다.
3. `C`, `epsilon`, `gamma` 같은 하이퍼파라미터가 결과에 영향을 준다.

In [ ]:
# 보스턴 집값 데이터 로드
boston_df = pd.read_csv('data/boston_housing_train.csv')

# MEDV를 타겟(집값)으로 사용하고, 나머지 컬럼은 입력 특성으로 사용한다.
X = boston_df.drop('MEDV', axis=1).to_numpy()
y = boston_df['MEDV'].to_numpy()

print(X.shape, y.shape)
print(boston_df.columns.tolist())
print(boston_df.head())

In [ ]:
# 데이터 준비
# SVR은 거리 기반 성격이 강하므로 스케일 차이에 민감하다.
# 특성의 범위가 크게 다르면 특정 특성이 거리 계산에 지나치게 큰 영향을 줄 수 있다.
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train_raw.shape, y_train_raw.shape)
print(X_test_raw.shape, y_test_raw.shape)

print('X 스케일링 전:', X_train_raw[:1])
print('y 스케일링 전:', y_train_raw[:3])

# 회귀에서도 X뿐 아니라 y까지 스케일링해서 학습하는 경우가 있다.
# 특히 SVR처럼 거리와 마진 개념을 쓰는 모델에서는 학습 안정성에 도움이 될 수 있다.
X_scaler = StandardScaler()
X_train = X_scaler.fit_transform(X_train_raw)
X_test = X_scaler.transform(X_test_raw)

y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train_raw.reshape(-1, 1)).ravel()
y_test = y_scaler.transform(y_test_raw.reshape(-1, 1)).ravel()

print('X 스케일링 후:', X_train[:1])
print('y 스케일링 후:', y_train[:3])

### MSE를 확인하는 이유

회귀에서는 정답/오답이 아니라 **얼마나 틀렸는가**가 중요하다.  
그래서 커널별로 예측 오차의 크기를 비교하기 위해 MSE를 사용한다.

다만 지금은 `y`를 스케일링한 상태이므로, 이 MSE는 스케일된 기준의 오차이다.  
실제 집값 단위 해석이 필요할 때는 다시 원래 단위로 복원해서 보는 것이 좋다.

## 03. 하이퍼파라미터 튜닝

SVR에서 자주 보는 핵심 파라미터는 다음과 같다.

- `C`: 튜브 밖 오차를 얼마나 강하게 벌줄지
- `epsilon`: 어느 정도 오차까지 허용할지
- `gamma`: RBF 커널이 데이터 변화에 얼마나 민감하게 반응할지

즉, SVR 튜닝은
"얼마나 엄격하게 맞출 것인가"
"얼마나 유연하게 곡선을 만들 것인가"
를 조절하는 과정이라고 볼 수 있다.

## 정리

1. SVR은 모든 오차를 똑같이 줄이려는 회귀가 아니다.
2. 일정 범위(`epsilon`) 안의 작은 오차는 허용하는 관점이 있다.
3. 선형/다항/RBF 커널에 따라 예측 곡선의 유연성이 달라진다.
4. SVR은 스케일링의 영향을 크게 받으므로 전처리가 중요하다.
5. `C`, `epsilon`, `gamma`를 조절하면서 오차와 일반화 성능의 균형을 맞춘다.